In [1]:
import numpy as np
import pandas as pd
import FastRolling as fr
import os
import polars as pl
from datetime import datetime
from StrategyAnalyzer import structure_stats, annual_stat

In [2]:
def timing1(pf, fret):
    med = 0.5 * (pf['high'] + pf['low'])
    f1 = pf['open'] - med
    f2 = pf['close'] - med
    f3 = pf['volume'].diff()
    c1 = (f1 < 0) & (f2 < 0)
    c2 = (f1 < 0) & (f2 > 0) & (f3 > 0)
    c3 = (f1 < 0) & (f2 > 0) & (f3 <= 0)
    c4 = (f1 > 0) & (f2 > 0)
    structure_stats(fret, *(c1, c2, c3, c4))

def timing2(pf, fret):
    window = 5
    f1 = pf['pct_change'].rolling(window).apply(lambda x: np.sign(x).sum(), raw=True)
    f2 = pf['pct_change'].rolling(window).sum()
    f3 = (pf['high'] - pf['close']) / (pf['close'] - pf['low'] + 1e-5)
    c1 = (f1*f2<0)
    c2 = (f1*f2>=0)&(f3 >= 1)
    c3 = (f1*f2>=0)&(f3 < 1)
    structure_stats(fret, *(c1, c2, c3))

def timing3(pf, fret):
    window = 6
    f1 = (pf['high'] + pf['low']) - (pf['close'] + pf['open'])
    f2 = pf['low'].rolling(5).skew()
    f3 = f1.rolling(2).sum()
    def calc_up_dn_ratio(x):
        up = x[x > 0]
        dn = x[x < 0]
        up_mean = up.mean() if len(up) > 0 else 1e-4
        dn_mean = abs(dn.mean()) if len(dn) > 0 else 1e-4
        return up_mean / dn_mean
    f4 = f1.rolling(window).apply(calc_up_dn_ratio, raw=False)
    c1 = (f1 >= 0) & (f4 >= 1)
    c2 = (~c1) & (f3 > 0) & (f2 > 0)
    c3 = (~c1) & (f3 < 0) & (f2 < 0)
    structure_stats(fret, *(c1, c2, c3))

def timing4(pf, fret):
    window = 10
    f1 = pf['low'].pct_change().rolling(window).sum()
    f2 = pf['low'].pct_change().apply(lambda x: np.sign(x)).rolling(window).sum()
    f3 = pf['high'].diff()
    f4 = pf['volume'].diff()
    c1 = (f1 > 0) & (f2 > 0)
    c2 = (~c1) & (f3 < 0)
    c3 = (~c1) & (f3 >= 0) & (f4 > 0)
    c4 = (~c1) & (f3 >= 0) & (f4 <= 0)
    structure_stats(fret, *(c1, c2, c3, c4))

def timing5(pf, fret):
    window = 10
    f1 = pf['pct_change'].rolling(window).std().diff()
    f2 = fr.sma(pf['volume'], window).diff()
    f3 = pf['skew']
    f4 = pf['extup'].diff()
    c1 = (f1 > 0) & (f2 > 0)
    c2 = (f1 < 0) & (f3 < 0)
    c3 = (~(c1 | c2)) & (f4 > 0)
    c4 = (~(c1 | c2)) & (f4 <= 0)
    structure_stats(fret, *(c1, c2, c3, c4))

def timing6(pf, fret):
    f1 = pf['skew'].diff()
    f2 = pf['high'].diff()
    c1 = (f1 > 0) & (f2 > 0)
    c2 = (f1 < 0) & (f2 < 0)
    structure_stats(fret, *(c1, c2))

def timing7(pf, fret):
    f1 = pf['closeret']
    profit_ratio = pf['closeret'].abs() / (pf['pct_change'].abs() + 1e-5)
    f2 = fr.signal_uniform(profit_ratio, 5)
    f3 = fr.signal_uniform(pf['closevol'], 5)
    c1 = (f1 > 0) & (f2 > 0)
    c2 = (f1 < 0) & (f3 > 0)
    structure_stats(fret, *(c1, c2))

In [24]:
from datetime import time
codes = ['hs300', 'zz500', 'zz1000']
code = codes[2]
start1 = '20180101'; end1 = '20240331'; start2 = '20240401'; end2 = '20260531'
pf = (pl.read_parquet(f'./idx_mbar/{code}.parquet')
      .with_columns([pl.col('close').cast(pl.Float64),
                     pl.col('open').cast(pl.Float64),
                     pl.col('high').cast(pl.Float64),
                     pl.col('low').cast(pl.Float64),
                     pl.col('pre').cast(pl.Float64),
                     pl.col('volume').cast(pl.Int64),
                     pl.col('amount').cast(pl.Int64)])
      .with_columns([(pl.col('close')/pl.col('pre')-1).alias('ret'),
                     pl.col('time_id').dt.time().alias('time'),
      ])
      .with_columns([pl.col('ret').mean().over('date_id').alias('mu'),
                     pl.col('ret').std().over('date_id').alias('sigma')])
      .group_by('date_id').agg([pl.col('ret').filter(pl.col('time')>=time(14, 30)).sum().alias('closeret'),
                                pl.col('close').skew().alias('skew'),
                                pl.col('ret').filter((pl.col('ret')-pl.col('mu'))>pl.col('sigma')).sum().alias('extup'),
                                pl.col('volume').filter(pl.col('time')>=time(14, 30)).sum().alias('closevol'),
                                ])
      ).sort('date_id').to_pandas().set_index('date_id')
pf_ = pd.read_pickle(f'./data/{code}.pkl')
pf_.index = pd.to_datetime(pf_.index)
pf = pd.concat([pf_, pf], axis=1)
pf.index = pf.index.strftime('%Y%m%d')

In [26]:
day = 1
pf1 = pf.loc[(pf.index<=end1)&(pf.index>=start1)]
pf2 = pf.loc[(pf.index<=end2)&(pf.index>=start2)]
fret1 = pf1['open'].pct_change(day).shift(-day-1)
fret2 = pf2['open'].pct_change(day).shift(-day-1)
timing7(pf1, fret1)
timing7(pf2, fret2)

   count     wgt   annret  annstd    winr     odd   kelly annsp
1    501  33.07%  -13.11%  13.11%  53.49%  1.1640  0.1576  1.00
2    440  29.04%   10.75%  14.24%  56.14%  1.0043  0.1251  0.75
   count     wgt   annret  annstd    winr     odd   kelly annsp
1    167  31.99%  -15.42%  14.91%  54.49%  1.1374  0.1647  1.03
2    143  27.39%   21.85%  13.05%  58.04%  1.2418  0.3012  1.67
